# Data Preprocessing - Text Classification

## Import Libraries, Variables and Helper Functions

In [1]:
## Add absolute path to this notebook
import sys
import os

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import random
import string
import re
import numpy as np
import pandas as pd

from src.configs.config import DEFAULT_CATEGORIES, STOPWORD_RATIOS

import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /home/hn-
[nltk_data]     minh/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/hn-
[nltk_data]     minh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
REMOVE_NUMBERS = True      # True: Xóa số, False: Giữ nguyên
APPLY_STEMMING = False     # True: Dùng PorterStemmer, False: Không dùng

# Mức độ loại bỏ Stopword: 0 (0%), 1 (20%), 2 (50%), 3 (80%), 4 (100%)
# Logic: Mức 0.0 sẽ không xóa gì.
# Các mức lớn hơn 0 sẽ sử dụng tập stopword kết hợp (NLTK + EDA)
# và dùng thông số này để lọc max_df trong TF-IDF (loại bỏ top % từ xuất hiện quá nhiều).
STOPWORD_LEVEL = STOPWORD_RATIOS[4]

## Load Data

In [4]:
train_data = fetch_20newsgroups(
    subset='train',
    categories=DEFAULT_CATEGORIES,
    remove=('headers', 'footers', 'quotes')
)

test_data = fetch_20newsgroups(
    subset='test',
    categories=DEFAULT_CATEGORIES,
    remove=('headers', 'footers', 'quotes')
)

train_df = pd.DataFrame({
    'text': train_data.data,
    'target': train_data.target,
    'target_name': [train_data.target_names[i] for i in train_data.target]
})

test_df = pd.DataFrame({
    'text': train_data.data,
    'target': train_data.target,
    'target_name': [train_data.target_names[i] for i in train_data.target]
})

print(f"Train shape: {len(train_df)}")
print(f"Number of classes: {len(train_data.target_names)}")

Train shape: 2242
Number of classes: 4


## Filter EDA Outliers

In [5]:
train_df['word_count'] = train_df['text'].apply(lambda x: len(str(x).split()))
train_df['char_count'] = train_df['text'].apply(lambda x: len(str(x)))

train_df = train_df[train_df['word_count'] > 0]

In [6]:
train_df['punct_count'] = train_df['text'].apply(lambda x: len([c for c in x if c in string.punctuation]))
train_df['punct_ratio'] = train_df['punct_count'] / (train_df['char_count'] + 1)
train_df = train_df[train_df['punct_ratio'] < 0.5]

print(f"Number of documents after cleaning: {len(train_df)}")

Number of documents after cleaning: 2178


## Text Preprocessing Function

In [7]:
base_stopwords = set(stopwords.words('english'))
custom_stopwords = {'don', 've', 'like', 'just'}
ALL_STOPWORDS = base_stopwords.union(custom_stopwords)

stemmer = PorterStemmer()

def preprocess_text(text, drop_rate=0.8, remove_num=False, apply_stem=False):
    text = str(text).lower()
    if remove_num:
        text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)

    tokens = word_tokenize(text)

    # Find all stopwords in the current document
    stopword_indices = [i for i, word in enumerate(tokens) if word in ALL_STOPWORDS]

    # Calculate how many stopwords to drop
    num_to_drop = int(len(stopword_indices) * drop_rate)

    # Randomly select indices of stopwords to drop
    drop_indices = set(random.sample(stopword_indices, num_to_drop))

    # Keep non-stopwords and the un-dropped stopwords
    final_tokens = []
    for i, token in enumerate(tokens):
        if i not in drop_indices:
            final_tokens.append(stemmer.stem(token) if apply_stem else token)

    return ' '.join(final_tokens)

# Apply preprocessing to train set
train_df['cleaned_text'] = train_df['text'].apply(
    lambda x: preprocess_text(x, drop_rate=STOPWORD_LEVEL, remove_num=REMOVE_NUMBERS, apply_stem=APPLY_STEMMING)
)

In [8]:
train_df[['text', 'cleaned_text']].sample(5)

,text,cleaned_text
903,I just got out of the Army. Go signal corps or...,got army go signal corps intelligence photoint...
2025,\n\nWrong information. They just announced tha...,wrong information announced suhonen made deal ...
1338,"Replying to A.J. Teel:\n\n\tWell, the two nift...",replying j teel well two nifty letters giving ...
684,\nDrive down to Cincinnati and take a look. N...,drive cincinnati take look pretty things much ...
610,"\nYes, I did punch in the wrong numbers (worki...",yes punch wrong numbers working many late nite...


## TF-IDF Vectorization

In [9]:
# Fit and transform TF-IDF Vectorizer on train set
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['cleaned_text'])
y_train = train_df['target'].values

print(f"Train TF-IDF shape: {X_train_tfidf.shape}")

Train TF-IDF shape: (2178, 25701)


## Load and Process Test Data

In [10]:
test_df['cleaned_text'] = test_df['text'].apply(
    lambda x: preprocess_text(x, drop_rate=1.0, remove_num=REMOVE_NUMBERS, apply_stem=APPLY_STEMMING)
)

X_test_tfidf = tfidf_vectorizer.transform(test_df['cleaned_text'])
y_test = test_df['target'].values

print(f"Test TF-IDF shape: {X_test_tfidf.shape}")

Test TF-IDF shape: (2242, 25701)


## Model Testing (Optional)

In [11]:
# nb_model = MultinomialNB()
# nb_model.fit(X_train_tfidf, y_train)

# y_pred = nb_model.predict(X_test_tfidf)

# accuracy = accuracy_score(y_test, y_pred)
# print(f"Accuracy: {accuracy:.4f}\n")

# print("Classification Report:")
# print(classification_report(y_test, y_pred, target_names=DEFAULT_CATEGORIES))

## Summary

1. **Training Data (Train):**
* `X_train_tfidf`: The feature matrix of the training set.
* `y_train`: The actual labels (target) of the training set.


2. **Evaluation Data (Test):**
* `X_test_tfidf`: The feature matrix of the test set (transformed using the training set's configuration).
* `y_test`: The actual labels (target) of the test set.


3. **Supplementary Information:**
* `DEFAULT_CATEGORIES`: A list of 4 topic names (used to map numerical labels to text in the Confusion Matrix / Classification Report).
* `tfidf_vectorizer` & `preprocess_text`: Retained for preprocessing and transforming text entered by users later (Inference phase).